<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/Context_Aware_Neural_Recommendation_Engine(week_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Install PySpark  (used to process large dataset)

In [2]:
!pip install -q pyspark

#Import PySpark  (let us to work with spark)

In [3]:
from pyspark.sql import SparkSession

#Create the Spark Session

In [4]:
spark = (
    SparkSession.builder
    .appName("HM-Recommendation-System")
    .getOrCreate()
)

In [ ]:
print("Spark Version:", spark.version)

Spark Version: 4.0.3


#Downloading the H&M Dataset from kaggle

In [ ]:
from google.colab import files
uploaded = files.upload()


In [11]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [12]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations

100% 28.7G/28.7G [04:23<00:00, 117MB/s]



In [13]:
!unzip -q h-and-m-personalized-fashion-recommendations.zip -d hm_dataset

In [14]:
import os

print(os.listdir("hm_dataset"))

['images', 'customers.csv', 'sample_submission.csv', 'transactions_train.csv', 'articles.csv']


#Loading the Dataset into PySpark

#Define the Dataset Path

In [15]:
DATA_PATH = "/content/hm_dataset"

#Load customers.csv

In [16]:
customers_df = spark.read.csv(
    f"{DATA_PATH}/customers.csv",
    header=True,
    inferSchema=True
)

#Load articles.csv

In [17]:
articles_df = spark.read.csv(
    f"{DATA_PATH}/articles.csv",
    header=True,
    inferSchema=True
)

#Load transactions_train.csv

In [18]:
transactions_df = spark.read.csv(
    f"{DATA_PATH}/transactions_train.csv",
    header=True,
    inferSchema=True
)

#Inspect the Data

In [ ]:
customers_df.show(5)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 5 rows


In [ ]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- club_member_status: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



In [ ]:
customers_df.count()

1371980

In [ ]:
articles_df.show(5)

+----------+------------+-----------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------------+--------------+----------------+----------+--------------------+----------------+------------------+--------------------+
|article_id|product_code|        prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|      index_name|index_group_no|index_group_name|section_no|        section_name|garment_group_no|garment_group_name|         detail_desc|
+----------+------------+-----------------+-------------

In [ ]:
articles_df.printSchema()

root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- index_group_name: string (nullable = true)

In [ ]:
articles_df.count()

105542

In [ ]:
transactions_df.show(5)

+----------+--------------------+----------+--------------------+----------------+
|     t_dat|         customer_id|article_id|               price|sales_channel_id|
+----------+--------------------+----------+--------------------+----------------+
|2018-09-20|000058a12d5b43e67...| 663713001|0.050830508474576264|               2|
|2018-09-20|000058a12d5b43e67...| 541518023| 0.03049152542372881|               2|
|2018-09-20|00007d2de826758b6...| 505221004| 0.01523728813559322|               2|
|2018-09-20|00007d2de826758b6...| 685687003|0.016932203389830508|               2|
|2018-09-20|00007d2de826758b6...| 685687004|0.016932203389830508|               2|
+----------+--------------------+----------+--------------------+----------------+
only showing top 5 rows


In [ ]:
transactions_df.printSchema()

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [ ]:
transactions_df.count()

31788324

#Missing Values in Customers

In [ ]:
from pyspark.sql.functions import col, count, when

customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

+-----------+------+------+------------------+----------------------+-----+-----------+
|customer_id|    FN|Active|club_member_status|fashion_news_frequency|  age|postal_code|
+-----------+------+------+------------------+----------------------+-----+-----------+
|          0|895050|907576|              6062|                 16009|15861|          0|
+-----------+------+------+------------------+----------------------+-----+-----------+



#Missing Values in Articles

In [ ]:
articles_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in articles_df.columns
]).show()

+----------+------------+---------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------+--------------+----------------+----------+------------+----------------+------------------+-----------+
|article_id|product_code|prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|index_name|index_group_no|index_group_name|section_no|section_name|garment_group_no|garment_group_name|detail_desc|
+----------+------------+---------+---------------+-----------------+------------------+-----------------------+------

#Missing Values in Transactions

In [ ]:
transactions_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in transactions_df.columns
]).show()

+-----+-----------+----------+-----+----------------+
|t_dat|customer_id|article_id|price|sales_channel_id|
+-----+-----------+----------+-----+----------------+
|    0|          0|         0|    0|               0|
+-----+-----------+----------+-----+----------------+



#inspect the unique values in customers

In [ ]:
customers_df.groupBy("club_member_status").count().orderBy("count", ascending=False).show()

+------------------+-------+
|club_member_status|  count|
+------------------+-------+
|            ACTIVE|1272491|
|        PRE-CREATE|  92960|
|              NULL|   6062|
|         LEFT CLUB|    467|
+------------------+-------+



In [ ]:
customers_df.groupBy("fashion_news_frequency").count().orderBy("count", ascending=False).show()

+----------------------+------+
|fashion_news_frequency| count|
+----------------------+------+
|                  NONE|877711|
|             Regularly|477416|
|                  NULL| 16009|
|               Monthly|   842|
|                  None|     2|
+----------------------+------+



In [ ]:
customers_df.groupBy("FN").count().show()

+----+------+
|  FN| count|
+----+------+
|NULL|895050|
| 1.0|476930|
+----+------+



In [ ]:
customers_df.groupBy("Active").count().show()

+------+------+
|Active| count|
+------+------+
|  NULL|907576|
|   1.0|464404|
+------+------+



#Clean the Customers Dataset

In [19]:
from pyspark.sql.functions import col, when

#Standardize fashion_news_frequency (we found NONE and None)

In [20]:
customers_df = customers_df.withColumn(
    "fashion_news_frequency",
    when(col("fashion_news_frequency") == "None", "NONE")
    .otherwise(col("fashion_news_frequency"))
)

#Fill Missing Values

In [21]:
customers_df = customers_df.fillna({
    "FN": 0,
    "Active": 0,
    "club_member_status": "UNKNOWN",
    "fashion_news_frequency": "NONE"
})

#Compute the Median Age

In [22]:
median_age = customers_df.approxQuantile("age", [0.5], 0.01)[0]

print("Median Age:", median_age)

Median Age: 32.0


#Fill Missing Ages

In [23]:
customers_df = customers_df.fillna({
    "age": median_age
})

#Verify the Cleaning

In [24]:
from pyspark.sql.functions import count, when

customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

+-----------+---+------+------------------+----------------------+---+-----------+
|customer_id| FN|Active|club_member_status|fashion_news_frequency|age|postal_code|
+-----------+---+------+------------------+----------------------+---+-----------+
|          0|  0|     0|                 0|                     0|  0|          0|
+-----------+---+------+------------------+----------------------+---+-----------+



In [25]:
from pyspark.sql.functions import col

customers_df = (
    customers_df
    .withColumn("FN", col("FN").cast("int"))
    .withColumn("Active", col("Active").cast("int"))
)

In [26]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: integer (nullable = true)
 |-- Active: integer (nullable = true)
 |-- club_member_status: string (nullable = false)
 |-- fashion_news_frequency: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



#Clean the Articles Dataset

In [27]:
articles_df = articles_df.fillna({
    "detail_desc": "No Description Available"
})

In [28]:
from pyspark.sql.functions import count, when, col

articles_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in articles_df.columns
]).show()

+----------+------------+---------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------+--------------+----------------+----------+------------+----------------+------------------+-----------+
|article_id|product_code|prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|index_name|index_group_no|index_group_name|section_no|section_name|garment_group_no|garment_group_name|detail_desc|
+----------+------------+---------+---------------+-----------------+------------------+-----------------------+------

#Check Duplicate Transactions

In [29]:
total_rows = transactions_df.count()
distinct_rows = transactions_df.distinct().count()

print("Total Rows :", total_rows)
print("Distinct Rows :", distinct_rows)
print("Duplicate Rows :", total_rows - distinct_rows)

Total Rows : 31788324
Distinct Rows : 28813419
Duplicate Rows : 2974905


#Find the Date Range

In [30]:
from pyspark.sql.functions import min, max

transactions_df.select(
    min("t_dat").alias("Start_Date"),
    max("t_dat").alias("End_Date")
).show()

+----------+----------+
|Start_Date|  End_Date|
+----------+----------+
|2018-09-20|2020-09-22|
+----------+----------+



#Count Unique Customers

In [31]:
from pyspark.sql.functions import countDistinct

transactions_df.select(
    countDistinct("customer_id").alias("Unique_Customers")
).show()

+----------------+
|Unique_Customers|
+----------------+
|         1362281|
+----------------+



#Count Unique Products

In [32]:
transactions_df.select(
    countDistinct("article_id").alias("Unique_Articles")
).show()

+---------------+
|Unique_Articles|
+---------------+
|         104547|
+---------------+



#Metric	Value
Total Transactions	31,788,324,
Distinct Transaction Rows	28,813,419,
Duplicate Records	2,974,905,
Purchase History	2 years,
Customers with Purchases	1,362,281,
Products Purchased	104,547

#Save the Cleaned Data

In [33]:
PROCESSED_PATH = "/content/drive/MyDrive/Recommendation_Engine/data/processed"

In [34]:
customers_df.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/customers_clean.parquet"
)

In [36]:
articles_df.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/articles_clean.parquet"
)

In [37]:
transactions_df.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/transactions_clean.parquet"
)

#Identify Cold-Start Users

In [38]:
from pyspark.sql.functions import col

# Customers who made at least one purchase
purchased_customers = transactions_df.select("customer_id").distinct()

# Customers with no purchase history
cold_start_users = customers_df.join(
    purchased_customers,
    on="customer_id",
    how="left_anti"
)

print("Cold Start Users:", cold_start_users.count())

Cold Start Users: 9699


In [39]:
cold_start_users.show(5)

+--------------------+---+------+------------------+----------------------+---+--------------------+
|         customer_id| FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+---+------+------------------+----------------------+---+--------------------+
|0033ed9017159dac2...|  1|     1|            ACTIVE|             Regularly| 20|2c29ae653a9282cce...|
|00f53046607c22cd6...|  0|     0|            ACTIVE|                  NONE| 17|d8983094daae78cec...|
|01587bfe37f402820...|  0|     0|            ACTIVE|                  NONE| 37|2c29ae653a9282cce...|
|0166011fdf2b9d363...|  1|     1|            ACTIVE|             Regularly| 52|73718e25b2ad62deb...|
|024d68550b2390c3c...|  1|     1|            ACTIVE|             Regularly| 32|2c29ae653a9282cce...|
+--------------------+---+------+------------------+----------------------+---+--------------------+
only showing top 5 rows


#Identify Cold-Start Items

In [40]:
purchased_articles = transactions_df.select("article_id").distinct()

cold_start_items = articles_df.join(
    purchased_articles,
    on="article_id",
    how="left_anti"
)

print("Cold Start Items:", cold_start_items.count())

Cold Start Items: 995


In [41]:
cold_start_items.select(
    "article_id",
    "prod_name",
    "product_type_name"
).show(5, truncate=False)

+----------+----------------------+------------------------+
|article_id|prod_name             |product_type_name       |
+----------+----------------------+------------------------+
|187949032 |Padded pyjama         |Pyjama jumpsuit/playsuit|
|288859020 |Kakan 2-p cableknit BG|Underwear Tights        |
|395730045 |VIOLA 2-pack (TVP)    |Polo shirt              |
|462435036 |6P Tanktop Body       |Bodysuit                |
|485689035 |Bobby l/l pj BB       |Pyjama set              |
+----------+----------------------+------------------------+
only showing top 5 rows


In [42]:
cold_start_users.write \
    .mode("overwrite") \
    .parquet(f"{PROCESSED_PATH}/cold_start_users.parquet")

In [43]:
cold_start_items.write \
    .mode("overwrite") \
    .parquet(f"{PROCESSED_PATH}/cold_start_items.parquet")

In [44]:
import os

print(os.listdir(PROCESSED_PATH))

['customers_clean.parquet', 'articles_clean.parquet', 'transactions_clean.parquet', 'cold_start_users.parquet', 'cold_start_items.parquet']
